# 06 Model Evaluation And Validation

Purpose: collect model quality, data-quality policy, and MVP limitations into one validation notebook.

In [1]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: D:\Project 1\rcm-cms-mvp


In [2]:
from src.models.evaluate import compile_validation_summary

validation = compile_validation_summary()
display(validation)

,area,check,status,result,decision
0,forecasting,best_holdout_model_observed_enrollment,pass,"linear_drift MAPE=0.0005, RMSE=22175.08",Use best MAPE for headline; keep baseline comparisons visible.
1,forecasting,best_holdout_model_proxy_revenue,pass,"linear_drift MAPE=0.0005, RMSE=2550134.20",Use best MAPE for headline; keep baseline comparisons visible.
2,prior_auth,seeded_demo_classifier,pass,"accuracy=0.933, roc_auc=0.959","Frame as CMS-aligned risk prioritization, not production payer benchmarking."
3,data_quality,outlier_policy,pass,national_monthly: 4; state_monthly: 238; county_monthly: 9542,Flag outliers; do not drop source rows.
4,data_quality,forecasting_readiness,pass,"29 months, 590398 rows preserved",Rows with CMS suppressed enrollment are retained; numeric totals use observed numeric enrollment and should be descr...


## Forecast Metrics

Why: compare all models, not just the selected model. This prevents overclaiming.

In [3]:
forecast_metrics = pd.read_csv(TABLE_DIR / "forecast_metrics.csv")
display(forecast_metrics)
fig = px.bar(forecast_metrics, x="model", y="mape", color="target", barmode="group", title="Forecast Holdout MAPE by Model")
fig.show()

,model,target,mae,rmse,mape,holdout_months
0,linear_drift,observed_enrollment,1.735513e+04,2.217508e+04,0.000482,3
1,exp_smoothing,observed_enrollment,9.985923e+04,1.012477e+05,0.002774,3
2,naive_last_value,observed_enrollment,1.934117e+05,2.013995e+05,0.005369,3
3,prophet,observed_enrollment,3.123028e+05,3.151121e+05,0.008671,3
4,moving_average_3m,observed_enrollment,3.982497e+05,4.021896e+05,0.011057,3
5,linear_drift,proxy_revenue,1.995840e+06,2.550134e+06,0.000482,3
6,exp_smoothing,proxy_revenue,8.223221e+06,8.723373e+06,0.001987,3
7,naive_last_value,proxy_revenue,2.224234e+07,2.316094e+07,0.005369,3
8,prophet,proxy_revenue,3.591483e+07,3.623789e+07,0.008671,3
9,moving_average_3m,proxy_revenue,4.579871e+07,4.625181e+07,0.011057,3


## PA Metrics

Why: classifier quality is reported separately from the public-data caveat.

In [4]:
pa_metrics = pd.read_csv(TABLE_DIR / "pa_model_metrics.csv")
cm = pd.read_csv(TABLE_DIR / "pa_confusion_matrix.csv", index_col=0)
display(pa_metrics)
display(cm)

,model,accuracy,roc_auc,precision_approved,recall_approved,f1_approved,test_rows,note
0,gradient_boosting_pa_demo,0.933333,0.959135,0.961538,0.961538,0.961538,60,"Seeded public-data demo because CMS public PA files provide reporting schema, not request-level labels."


,pred_denied,pred_approved
actual_denied,6,2
actual_approved,2,50


## Data Quality Policy

Why: this MVP keeps data-quality decisions explicit: suppressed rows and outliers are flagged, not deleted.

In [5]:
display(pd.read_csv(TABLE_DIR / "ma_scp_missing_value_profile.csv"))
display(pd.read_csv(TABLE_DIR / "ma_scp_outlier_handling_summary.csv"))

,field,missing_or_unusable_rows,row_share
0,state,0,0.000000
1,county,0,0.000000
2,plan_type,0,0.000000
3,fips_code,0,0.000000
4,enrolled_raw,0,0.000000
5,enrolled,291362,0.493501


,level,rows_evaluated,iqr_outliers,action
0,national_monthly,29,4,flag_only_keep_rows
1,state_monthly,1568,238,flag_only_keep_rows
2,county_monthly,91504,9542,flag_only_keep_rows
